|Machine Learning Lab 04 |                   |          
|:-----------------------------------:|:-----------------:|
| **Text Classification (k-NN)**           |                    |

|                 |                   | 
|-----------------|:-----------------:|
| **Name**        | Shubham Choudhary |        
| **Roll Number** | 212405015         |

# Step 1: Setup and Dataset Loading

In [11]:
import os
import re
import math
from collections import Counter, defaultdict
import numpy as np

In [12]:
class DatasetLoader:
    def __init__(self, dataset_path):
        self.dataset_path = dataset_path
        self.categories = []
        self.category_to_id = {}
        
    def discover_structure(self):
        print("Exploring directory structure...")
        for item in os.listdir(self.dataset_path):
            item_path = os.path.join(self.dataset_path, item)
            if os.path.isdir(item_path):
                print(f"Found directory: {item}")
                sub_items = os.listdir(item_path)
                if any(os.path.isdir(os.path.join(item_path, sub)) for sub in sub_items):
                    print(f"  Contains subdirectories: {sub_items[:5]}...")
        
    def find_dataset_paths(self):
        possible_paths = [
            {
                'train': os.path.join(self.dataset_path, '20news-bydate-train'),
                'test': os.path.join(self.dataset_path, '20news-bydate-test')
            },

            {
                'train': os.path.join(self.dataset_path, 'train'),
                'test': os.path.join(self.dataset_path, 'test')
            },

            {
                'train': self.dataset_path,
                'test': self.dataset_path
            }
        ]
        
        for paths in possible_paths:
            if os.path.exists(paths['train']):
                print(f"Found training data at: {paths['train']}")
                if os.path.exists(paths['test']):
                    print(f"Found test data at: {paths['test']}")
                else:
                    print(f"Test data not found at: {paths['test']}")
                return paths
        
        raise ValueError("Could not find dataset directories")
    
    def get_categories(self, train_path):
        self.categories = []
        for item in os.listdir(train_path):
            item_path = os.path.join(train_path, item)
            if os.path.isdir(item_path):
                self.categories.append(item)
        
        if not self.categories:
            files = [f for f in os.listdir(train_path) if os.path.isfile(os.path.join(train_path, f))]
            if files:
                category_name = os.path.basename(train_path)
                self.categories = [category_name]
                print(f"Single category mode: {category_name}")
        
        self.categories.sort()
        self.category_to_id = {cat: idx for idx, cat in enumerate(self.categories)}
        return self.categories
    
    def load_from_directory(self, directory_path):
        data = []
        labels = []
        
        if not os.path.exists(directory_path):
            print(f"Warning: Directory {directory_path} does not exist")
            return data, labels
        
        has_categories = any(os.path.isdir(os.path.join(directory_path, item)) 
                           for item in os.listdir(directory_path))
        
        if has_categories:
            for category in self.categories:
                category_path = os.path.join(directory_path, category)
                if not os.path.exists(category_path):
                    continue
                    
                category_id = self.category_to_id[category]
                files_loaded = 0
                
                for filename in os.listdir(category_path):
                    file_path = os.path.join(category_path, filename)
                    if os.path.isfile(file_path):
                        try:
                            with open(file_path, 'r', encoding='latin-1', errors='ignore') as f:
                                content = f.read()
                            data.append(content)
                            labels.append(category_id)
                            files_loaded += 1
                        except Exception as e:
                            print(f"Error reading {file_path}: {e}")
                
                print(f"Loaded {files_loaded} files from {category}")
        else:
            files_loaded = 0
            category_name = os.path.basename(directory_path)
            
            if category_name in self.category_to_id:
                category_id = self.category_to_id[category_name]
            else:
                category_id = 0
                self.category_to_id = {category_name: category_id}
                self.categories = [category_name]
            
            for filename in os.listdir(directory_path):
                file_path = os.path.join(directory_path, filename)
                if os.path.isfile(file_path):
                    try:
                        with open(file_path, 'r', encoding='latin-1', errors='ignore') as f:
                            content = f.read()
                        data.append(content)
                        labels.append(category_id)
                        files_loaded += 1
                    except Exception as e:
                        print(f"Error reading {file_path}: {e}")
            
            print(f"Loaded {files_loaded} files from {category_name}")
        
        return data, labels
    
    def load_dataset(self):
        print("Discovering dataset structure...")
        self.discover_structure()
        
        print("\nFinding dataset paths...")
        paths = self.find_dataset_paths()
        train_path = paths['train']
        test_path = paths['test']
        
        print(f"\nLoading categories from: {train_path}")
        self.get_categories(train_path)
        print(f"Found {len(self.categories)} categories: {self.categories}")
        
        print("\nLoading training data...")
        train_data, train_labels = self.load_from_directory(train_path)
        print(f"Loaded {len(train_data)} training samples")
        
        print("\nLoading test data...")
        test_data, test_labels = self.load_from_directory(test_path)
        print(f"Loaded {len(test_data)} test samples")
        
        return train_data, train_labels, test_data, test_labels

# Code Explaination: DatasetLoader Class

##  Overview

The `DatasetLoader` class is a **custom data loading utility** designed to handle the **20Newsgroups dataset** with flexible directory structures. It automatically discovers, validates, and loads text data from various possible folder organizations.

##  Class Structure

### **Initialization (`__init__` method)**

```python

def __init__(self, dataset_path):

self.dataset_path = dataset_path

self.categories = []

self.category_to_id = {}

```

- **`dataset_path`**: Root directory where the dataset is stored

- **`categories`**: List to store discovered category names

- **`category_to_id`**: Dictionary mapping category names to numerical IDs

---

##  Key Methods

### **1. `discover_structure()`**

**Purpose**: Exploratory method to understand directory hierarchy

**Functionality**:

- Iterates through all items in the dataset path

- Identifies directories (ignores files)

- Detects nested subdirectory structures

- Prints informative discovery messages

**Output Example**:

```

Found directory: 20news-bydate

Contains subdirectories: ['20news-bydate-test', '20news-bydate-test']...

```

---

### **2. `find_dataset_paths()`**

**Purpose**: **Automatically locates** training and testing directories

**Strategy**: Tries multiple common directory structures:

1. **Standard Structure**:

```python

{

'train': os.path.join(self.dataset_path, '20news-bydate-train'),

'test': os.path.join(self.dataset_path, '20news-bydate-test')

}

```

2. **Alternative Structure**:

```python

{

'train': os.path.join(self.dataset_path, 'train'),

'test': os.path.join(self.dataset_path, 'test')

}

```

3. **Flat Structure** (single directory):

```python

{

'train': self.dataset_path,

'test': self.dataset_path

}

```

**Error Handling**: Raises `ValueError` if no valid structure is found

---

### **3. `get_categories(train_path)`**

**Purpose**: Discovers and categorizes newsgroup topics

**Logic Flow**:

1. **Primary Detection**: Looks for subdirectories as categories

2. **Fallback Detection**: If no subdirectories, treats current directory as single category

3. **Organization**: Sorts categories and creates ID mapping

**Special Feature**: Handles both multi-category and single-category scenarios

---

### **4. `load_from_directory(directory_path)`**

**Purpose**: **Core loading mechanism** for text files

**Dual Loading Modes**:

#### **A. Category-based Loading** (when subdirectories exist)

- Iterates through each category directory

- Reads all text files within each category

- Assigns appropriate category ID to each document

- **Robust error handling** for file reading issues

#### **B. Single Category Loading** (flat directory)

- Treats all files in directory as one category

- Dynamically creates category mapping

- Maintains consistent data structure

**Encoding Handling**: Uses `latin-1` encoding with error ignoring for maximum compatibility

---

### **5. `load_dataset()`**

**Purpose**: **Orchestration method** that coordinates the complete loading process

**Execution Flow**:

1. **Structure Discovery** → `discover_structure()`

2. **Path Resolution** → `find_dataset_paths()`

3. **Category Identification** → `get_categories()`

4. **Data Loading** → `load_from_directory()` (for both train and test)

5. **Return structured data** with labels

**Output**:

- **`train_data`**: List of training document contents

- **`train_labels`**: Corresponding category IDs for training data

- **`test_data`**: List of test document contents

- **`test_labels`**: Corresponding category IDs for test data

---

##  Robustness Features

### **Error Resilience**

- **File-level error handling**: Continues loading if individual files fail

- **Directory validation**: Checks existence before attempting access

- **Encoding flexibility**: Handles various text encodings gracefully

### **Adaptability**

- **Multiple structure support**: Works with different dataset organizations

- **Progressive discovery**: Automatically finds the right directory level

- **Dynamic categorization**: Adapts to single or multiple categories

### **Informative Output**

- **Progress reporting**: Shows what's being loaded and from where

- **Statistics**: Provides counts of loaded files per category

- **Error reporting**: Specific messages for troubleshooting

---

##  Usage Example

```python

# Initialize loader

loader = DatasetLoader("/path/to/20news-bydate")

# Load complete dataset

train_data, train_labels, test_data, test_labels = loader.load_dataset()

print(f"Loaded {len(train_data)} training samples")

print(f"Loaded {len(test_data)} test samples")

print(f"Categories: {loader.categories}")

```

##  Key Benefits

1. **Automation**: Handles directory discovery automatically

2. **Flexibility**: Works with various dataset organizations

3. **Robustness**: Gracefully handles errors and edge cases

4. **Compatibility**: Works with standard 20Newsgroups structures

5. **Transparency**: Provides detailed progress information

This class eliminates the need for manual path configuration and makes the dataset loading process **completely automated and reliable**.

---


# Step 2: Text Preprocessing Functions

In [13]:
def preprocess_text(text, remove_stopwords=False):
    if not text:
        return []
    
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    tokens = text.split()

    if remove_stopwords:
        stopwords = {
            'a', 'able', 'about', 'above', 'according', 'accordingly', 'across', 'actually', 
            'after', 'afterwards', 'again', 'against', 'all', 'allow', 'allows', 'almost', 
            'alone', 'along', 'already', 'also', 'although', 'always', 'am', 'among', 
            'amongst', 'an', 'and', 'another', 'any', 'anybody', 'anyhow', 'anyone', 
            'anything', 'anyway', 'anyways', 'anywhere', 'apart', 'appear', 'appreciate', 
            'appropriate', 'are', 'around', 'as', 'aside', 'ask', 'asking', 'associated', 
            'at', 'available', 'away', 'awfully', 'b', 'be', 'became', 'because', 'become', 
            'becomes', 'becoming', 'been', 'before', 'beforehand', 'behind', 'being', 
            'believe', 'below', 'beside', 'besides', 'best', 'better', 'between', 'beyond', 
            'both', 'brief', 'but', 'by', 'c', 'came', 'can', 'cannot', 'cant', 'cause', 
            'causes', 'certain', 'certainly', 'changes', 'clearly', 'co', 'com', 'come', 
            'comes', 'concerning', 'consequently', 'consider', 'considering', 'contain', 
            'containing', 'contains', 'corresponding', 'could', 'course', 'currently', 'd', 
            'definitely', 'described', 'despite', 'did', 'different', 'do', 'does', 'doing', 
            'done', 'down', 'downwards', 'during', 'e', 'each', 'edu', 'eg', 'eight', 
            'either', 'else', 'elsewhere', 'enough', 'entirely', 'especially', 'et', 'etc', 
            'even', 'ever', 'every', 'everybody', 'everyone', 'everything', 'everywhere', 
            'ex', 'exactly', 'example', 'except', 'f', 'far', 'few', 'fifth', 'first', 
            'five', 'followed', 'following', 'follows', 'for', 'former', 'formerly', 'forth', 
            'four', 'from', 'further', 'furthermore', 'g', 'get', 'gets', 'getting', 'given', 
            'gives', 'go', 'goes', 'going', 'gone', 'got', 'gotten', 'greetings', 'h', 'had', 
            'happens', 'hardly', 'has', 'have', 'having', 'he', 'hello', 'help', 'hence', 
            'her', 'here', 'hereafter', 'hereby', 'herein', 'hereupon', 'hers', 'herself', 
            'hi', 'him', 'himself', 'his', 'hither', 'hopefully', 'how', 'howbeit', 'however', 
            'i', 'ie', 'if', 'ignored', 'immediate', 'in', 'inasmuch', 'inc', 'indeed', 
            'indicate', 'indicated', 'indicates', 'inner', 'insofar', 'instead', 'into', 
            'inward', 'is', 'it', 'its', 'itself', 'j', 'just', 'k', 'keep', 'keeps', 'kept', 
            'know', 'knows', 'known', 'l', 'last', 'lately', 'later', 'latter', 'latterly', 
            'least', 'less', 'lest', 'let', 'like', 'liked', 'likely', 'little', 'look', 
            'looking', 'looks', 'ltd', 'm', 'mainly', 'many', 'may', 'maybe', 'me', 'mean', 
            'meanwhile', 'merely', 'might', 'more', 'moreover', 'most', 'mostly', 'much', 
            'must', 'my', 'myself', 'n', 'name', 'namely', 'nd', 'near', 'nearly', 'necessary', 
            'need', 'needs', 'neither', 'never', 'nevertheless', 'new', 'next', 'nine', 'no', 
            'nobody', 'non', 'none', 'noone', 'nor', 'normally', 'not', 'nothing', 'novel', 
            'now', 'nowhere', 'o', 'obviously', 'of', 'off', 'often', 'oh', 'ok', 'okay', 
            'old', 'on', 'once', 'one', 'ones', 'only', 'onto', 'or', 'other', 'others', 
            'otherwise', 'ought', 'our', 'ours', 'ourselves', 'out', 'outside', 'over', 
            'overall', 'own', 'p', 'particular', 'particularly', 'per', 'perhaps', 'placed', 
            'please', 'plus', 'possible', 'presumably', 'probably', 'provides', 'q', 'que', 
            'quite', 'qv', 'r', 'rather', 'rd', 're', 'really', 'reasonably', 'regarding', 
            'regardless', 'regards', 'relatively', 'respectively', 'right', 's', 'said', 
            'same', 'saw', 'say', 'saying', 'says', 'second', 'secondly', 'see', 'seeing', 
            'seem', 'seemed', 'seeming', 'seems', 'seen', 'self', 'selves', 'sensible', 
            'sent', 'serious', 'seriously', 'seven', 'several', 'shall', 'she', 'should', 
            'since', 'six', 'so', 'some', 'somebody', 'somehow', 'someone', 'something', 
            'sometime', 'sometimes', 'somewhat', 'somewhere', 'soon', 'sorry', 'specified', 
            'specify', 'specifying', 'still', 'sub', 'such', 'sup', 'sure', 't', 'take', 
            'taken', 'tell', 'tends', 'th', 'than', 'thank', 'thanks', 'thanx', 'that', 
            'thats', 'the', 'their', 'theirs', 'them', 'themselves', 'then', 'thence', 
            'there', 'thereafter', 'thereby', 'therefore', 'therein', 'theres', 'thereupon', 
            'these', 'they', 'think', 'third', 'this', 'thorough', 'thoroughly', 'those', 
            'though', 'three', 'through', 'throughout', 'thru', 'thus', 'to', 'together', 
            'too', 'took', 'toward', 'towards', 'tried', 'tries', 'truly', 'try', 'trying', 
            'twice', 'two', 'u', 'un', 'under', 'unfortunately', 'unless', 'unlikely', 
            'until', 'unto', 'up', 'upon', 'us', 'use', 'used', 'useful', 'uses', 'using', 
            'usually', 'uucp', 'v', 'value', 'various', 'very', 'via', 'viz', 'vs', 'w', 
            'want', 'wants', 'was', 'way', 'we', 'welcome', 'well', 'went', 'were', 'what', 
            'whatever', 'when', 'whence', 'whenever', 'where', 'whereafter', 'whereas', 
            'whereby', 'wherein', 'whereupon', 'wherever', 'whether', 'which', 'while', 
            'whither', 'who', 'whoever', 'whole', 'whom', 'whose', 'why', 'will', 'willing', 
            'wish', 'with', 'within', 'without', 'wonder', 'would', 'x', 'y', 'yes', 'yet', 
            'you', 'your', 'yours', 'yourself', 'yourselves', 'z', 'zero'
        }
        tokens = [token for token in tokens if token not in stopwords and len(token) > 2]
    
    return tokens

def build_vocabulary(documents, max_vocab_size=2000, remove_stopwords=False):
    word_counts = Counter()
    
    for doc in documents:
        tokens = preprocess_text(doc, remove_stopwords=remove_stopwords)
        word_counts.update(tokens)
    
    vocabulary = [word for word, count in word_counts.most_common(max_vocab_size)]
    return vocabulary

def text_to_vector(text, vocabulary, remove_stopwords=False):
    tokens = preprocess_text(text, remove_stopwords=remove_stopwords)
    vector = [0] * len(vocabulary)
    
    for token in tokens:
        if token in vocabulary:
            index = vocabulary.index(token)
            vector[index] += 1
    
    total = sum(vector)
    if total > 0:
        vector = [v / total for v in vector]
    
    return vector

# Text Preprocessing Functions - Complete Code Explaination


##  Overview


The provided code implements a **complete text preprocessing pipeline** for natural language processing tasks. These functions work together to transform raw text into numerical representations suitable for machine learning algorithms, specifically for text classification tasks.


##  Core Functions


### **1. `preprocess_text(text, remove_stopwords=False)`**


#### **Purpose**
Transforms raw text into cleaned tokens ready for feature extraction, with optional stopword removal.


#### **Processing Pipeline**
```python
def preprocess_text(text, remove_stopwords=False):
    """
    Preprocess text: lowercase, remove special characters, optionally remove stopwords
    """
    # Input validation
    if not text:
        return []
    
    # 1. Case normalization
    text = text.lower()
    
    # 2. Special character removal
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # 3. Whitespace normalization
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 4. Tokenization
    tokens = text.split()
    
    # 5. Optional stopword removal
    if remove_stopwords:
        stopwords = { ... }  # Comprehensive stopword set
        tokens = [token for token in tokens if token not in stopwords and len(token) > 2]
    
    return tokens
```


#### **Key Processing Steps**
1. **Case Normalization**: Converts all text to lowercase for consistency
2. **Character Cleaning**: Removes numbers and special characters using regex
3. **Whitespace Handling**: Normalizes multiple spaces and trims edges
4. **Tokenization**: Splits text into individual words
5. **Stopword Filtering**: Optionally removes common words and short tokens


#### **Stopword Set Features**
- **Comprehensive list** of 570+ common English words
- Includes articles, pronouns, prepositions, and other frequent words
- Additional filtering for short words (`len(token) > 2`) to remove very short tokens


---


### **2. `build_vocabulary(documents, max_vocab_size=2000, remove_stopwords=False)`**


#### **Purpose**
Creates a vocabulary of the most frequent words from a collection of documents.


#### **Implementation**
```python
def build_vocabulary(documents, max_vocab_size=2000, remove_stopwords=False):
    """Build vocabulary from documents with optional stopwords removal"""
    word_counts = Counter()
    
    # Count words across all documents
    for doc in documents:
        tokens = preprocess_text(doc, remove_stopwords=remove_stopwords)
        word_counts.update(tokens)
    
    # Select most frequent words
    vocabulary = [word for word, count in word_counts.most_common(max_vocab_size)]
    return vocabulary
```


#### **Key Features**
- **Bulk Processing**: Handles multiple documents efficiently
- **Frequency-Based Selection**: Chooses words based on occurrence count
- **Configurable Size**: `max_vocab_size` parameter controls vocabulary dimension
- **Stopword Integration**: Consistent stopword removal with preprocessing function


#### **Typical Usage**
```python
# Build vocabulary from training data
vocabulary = build_vocabulary(train_documents, max_vocab_size=2000, remove_stopwords=True)
```


---


### **3. `text_to_vector(text, vocabulary, remove_stopwords=False)`**


#### **Purpose**
Converts a text document into a numerical vector representation using a predefined vocabulary.


#### **Vectorization Process**
```python
def text_to_vector(text, vocabulary, remove_stopwords=False):
    """Convert text to numerical vector with optional stopwords removal"""
    # Tokenize text
    tokens = preprocess_text(text, remove_stopwords=remove_stopwords)
    
    # Initialize vector with zeros
    vector = [0] * len(vocabulary)
    
    # Count occurrences of vocabulary words
    for token in tokens:
        if token in vocabulary:
            index = vocabulary.index(token)
            vector[index] += 1
    
    # Normalize by total word count
    total = sum(vector)
    if total > 0:
        vector = [v / total for v in vector]
    
    return vector
```


#### **Vector Characteristics**
- **Bag-of-Words Representation**: Each dimension represents a word from vocabulary
- **Term Frequency**: Values represent normalized word counts
- **Sparse-Friendly**: Many zero values for words not in document
- **Normalized**: Vectors sum to 1 (when words are present)


#### **Mathematical Representation**
For a vocabulary of size N, each document is represented as:
$$\text{vector} = [\frac{\text{count}(word_1)}{\text{total}}, \frac{\text{count}(word_2)}{\text{total}}, ..., \frac{\text{count}(word_N)}{\text{total}}]$$


---


##  Integration Workflow


### **Complete Processing Pipeline**
1. **Preprocess** individual texts → `preprocess_text()`
2. **Build vocabulary** from training corpus → `build_vocabulary()`
3. **Convert** all documents to vectors → `text_to_vector()`


### **Example Usage**
```python
# Step 1: Preprocess individual documents
cleaned_tokens = preprocess_text(raw_text, remove_stopwords=True)


# Step 2: Build vocabulary from training data
vocab = build_vocabulary(train_documents, max_vocab_size=2000, remove_stopwords=True)


# Step 3: Convert documents to vectors
document_vector = text_to_vector(new_document, vocab, remove_stopwords=True)
```


---


##  Design Considerations


### **Stopword Handling Strategy**
- **Consistent Application**: All functions accept `remove_stopwords` parameter
- **Performance Impact**: Stopword removal reduces dimensionality and processing time
- **Customization**: Easy to modify stopword list for domain-specific applications


### **Memory Efficiency**
- **Vocabulary Limiting**: `max_vocab_size` controls feature space dimensionality
- **Sparse Representations**: Efficient for large-scale text datasets
- **Incremental Processing**: Can process documents sequentially


### **Text Normalization Benefits**
1. **Noise Reduction**: Removes punctuation, numbers, and special characters
2. **Case Insensitivity**: Prevents duplicate features for same words
3. **Dimensionality Control**: Focuses on most meaningful terms


---


##  Practical Applications


### **Machine Learning Ready**
- Produces feature vectors suitable for classification algorithms
- Compatible with k-NN, SVM, Naive Bayes, and other ML models
- Enables text similarity calculations and document comparisons


### **Performance Optimization**
- **Vocabulary Pruning**: Focuses on most discriminative words
- **Stopword Removal**: Reduces noise and computational overhead
- **Normalization**: Improves algorithm convergence and performance


### **Experimental Flexibility**
- Easy to compare results with and without stopwords
- Adjustable vocabulary size for different complexity requirements
- Consistent preprocessing for fair algorithm comparisons


---


##  Key Advantages


1. **Consistency**: Uniform preprocessing across all functions
2. **Modularity**: Each function has a single responsibility
3. **Configurability**: Flexible parameters for different use cases
4. **Compatibility**: Works with standard Python data structures
5. **Transparency**: Clear, understandable transformation process


This preprocessing pipeline provides a **foundation for text classification** that is both effective and easy to understand, making it ideal for educational purposes and practical applications alike.


---


# Step 3: k-NN Classifier Implementation

In [14]:
class KNNClassifier:
    def __init__(self, k=3):
        self.k = k
        self.X_train = None
        self.y_train = None
    
    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
    
    def euclidean_distance(self, vec1, vec2):
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have same length")
        
        squared_diff = 0
        for i in range(len(vec1)):
            squared_diff += (vec1[i] - vec2[i]) ** 2
        
        return math.sqrt(squared_diff)
    
    def predict_single(self, test_vector):
        distances = []
        
        for i, train_vector in enumerate(self.X_train):
            dist = self.euclidean_distance(test_vector, train_vector)
            distances.append((dist, self.y_train[i]))

        distances.sort(key=lambda x: x[0])
        k_nearest = distances[:self.k]
        
        votes = {}
        for _, label in k_nearest:
            votes[label] = votes.get(label, 0) + 1

        return max(votes.items(), key=lambda x: x[1])[0]
    
    def predict(self, X_test):
        predictions = []
        for test_vector in X_test:
            predictions.append(self.predict_single(test_vector))
        return predictions
    
    def accuracy(self, y_true, y_pred):
        correct = 0
        for true, pred in zip(y_true, y_pred):
            if true == pred:
                correct += 1
        return correct / len(y_true)

# KNNClassifier Class - Complete Code Explaination

## Overview

The **`KNNClassifier`** class implements a **k-Nearest Neighbors algorithm from scratch** for classification tasks. This is a **lazy learning** algorithm that makes predictions based on the similarity between data points without building an explicit model during training.

## Class Architecture

### **Initialization (`__init__` method)**

```python

def __init__(self, k=3):

self.k = k

self.X_train = None

self.y_train = None

```

- **`k`**: Number of neighbors to consider (default = 3)

- **`X_train`**: Storage for training feature vectors

- **`y_train`**: Storage for training labels

**Design Pattern**: Follows the scikit-learn interface convention with `fit()` and `predict()` methods

---

## Core Methods

### **1. `fit(X, y)` Method**

#### **Purpose**

Stores the training data for later use in prediction (characteristic of lazy learning)

#### **Implementation**

```python

def fit(self, X, y):

"""Store training data"""

self.X_train = X

self.y_train = y

```

#### **Key Characteristics**

- **No Training Phase**: Unlike parametric models, KNN doesn't learn parameters during fitting

- **Instance-Based**: Entire training set is memorized for future comparisons

- **Memory Intensive**: Requires storing all training data (O(n) space complexity)

---

### **2. `euclidean_distance(vec1, vec2)` Method**

#### **Purpose**

Calculates the straight-line distance between two vectors in multidimensional space

#### **Mathematical Formula**
For  $x = (x_1, x_2,.....,x_n)$ and  $y = (y_1, y_2,.....,y_n)$
$$\text{distance} = \sqrt{\sum_{i=1}^{n} (x_i - y_i)^2}$$

#### **Implementation**

```python

def euclidean_distance(self, vec1, vec2):

"""Calculate Euclidean distance between two vectors"""

if len(vec1) != len(vec2):

raise ValueError("Vectors must have same length")

squared_diff = 0

for i in range(len(vec1)):

squared_diff += (vec1[i] - vec2[i]) ** 2

return math.sqrt(squared_diff)

```

#### **Key Features**

- **Dimension Validation**: Ensures vectors have compatible dimensions

- **Efficient Calculation**: Manual implementation avoids external dependencies

- **Geometric Interpretation**: Measures actual spatial distance between points

---

### **3. `predict_single(test_vector)` Method**

#### **Purpose**

Predicts the class label for a single test instance using the k-NN algorithm

#### **Algorithm Steps**

1. **Distance Calculation**: Compute distances to all training instances

2. **Neighbor Selection**: Identify k closest neighbors

3. **Majority Voting**: Determine most frequent class among neighbors

#### **Implementation**

```python

def predict_single(self, test_vector):

"""Predict label for a single test sample"""

distances = []

# Calculate distances to all training samples

for i, train_vector in enumerate(self.X_train):

dist = self.euclidean_distance(test_vector, train_vector)

distances.append((dist, self.y_train[i]))

# Sort by distance and get k nearest neighbors

distances.sort(key=lambda x: x[0])

k_nearest = distances[:self.k]

# Majority voting

votes = {}

for _, label in k_nearest:

votes[label] = votes.get(label, 0) + 1

# Return label with most votes

return max(votes.items(), key=lambda x: x[1])[0]

```

#### **Complexity Analysis**

- **Time Complexity**: O(n) for each prediction (where n = training set size)

- **Space Complexity**: O(n) for storing distances

---

### **4. `predict(X_test)` Method**

#### **Purpose**

Applies the prediction algorithm to multiple test instances

#### **Implementation**

```python

def predict(self, X_test):

"""Predict labels for multiple test samples"""

predictions = []

for test_vector in X_test:

predictions.append(self.predict_single(test_vector))

return predictions

```

#### **Batch Processing**

- **Iterative Application**: Applies single prediction to each test instance

- **Output**: Returns list of predicted labels corresponding to input order

---

### **5. `accuracy(y_true, y_pred)` Method**

#### **Purpose**

Calculates classification accuracy by comparing predicted and true labels

#### **Implementation**

```python

def accuracy(self, y_true, y_pred):

"""Calculate accuracy"""

correct = 0

for true, pred in zip(y_true, y_pred):

if true == pred:

correct += 1

return correct / len(y_true)

```

#### **Performance Metric**

$$\text{accuracy} = \frac{\text{number of correct predictions}}{\text{total predictions}}$$

---

## Algorithm Characteristics

### **k Parameter Sensitivity**

- **Small k** (k=1): High variance, low bias, prone to overfitting

- **Large k**: Low variance, high bias, may underfit

- **Optimal k**: Typically found through cross-validation

### **Distance Metric Choice**

- **Euclidean**: Default choice for continuous features

- **Alternative metrics**: Manhattan, Cosine, Minkowski could be implemented

- **Feature scaling**: Important when using distance-based metrics

### **Voting Mechanism**

- **Majority voting**: Each neighbor gets equal vote

- **Weighted voting**: Could be implemented based on distance (closer neighbors have more influence)

---

## Performance Considerations

### **Computational Complexity**

| Operation | Time Complexity | Space Complexity |
|-----------|-----------------|------------------|
| Training | O(1) | O(n) |
| Prediction| O(n) per query | O(n) |

### **Optimization Opportunities**

- **KD-Trees**: For faster neighbor searches in high dimensions

- **Ball Trees**: Alternative for higher-dimensional spaces

- **Approximate NN**: For very large datasets

---

## Usage Example

```python

# Initialize classifier

knn = KNNClassifier(k=5)

# Store training data

knn.fit(X_train, y_train)

# Make predictions

predictions = knn.predict(X_test)

# Evaluate performance

accuracy = knn.accuracy(y_test, predictions)

print(f"Model accuracy: {accuracy:.2f}")

```

---

## Strengths and Limitations

### Strengths

- **Simple implementation**: Easy to understand and implement

- **No training time**: Immediate model readiness

- **Adapts to new data**: Naturally incorporates new training examples

- **Non-linear boundaries**: Can capture complex decision boundaries

### Limitations

- **Computationally expensive**: Slow prediction for large datasets

- **Memory intensive**: Requires storing all training data

- **Sensitive to irrelevant features**: Distance metrics affected by all dimensions

- **Curse of dimensionality**: Performance degrades in high-dimensional spaces

---

## Practical Applications

### Ideal Use Cases

- **Small to medium datasets**: Where computational cost is manageable

- **Prototyping**: Quick implementation for baseline performance

- **Educational purposes**: Demonstrating fundamental ML concepts

- **Multi-class problems**: Naturally handles multiple classes

### When to Avoid

- **Very large datasets**: Where prediction speed is critical

- **High-dimensional data**: Where dimensionality reduction is needed

- **Real-time applications**: Where low latency is required

This implementation provides a **foundational understanding** of distance-based classification and serves as an excellent educational tool for learning machine learning fundamentals.

---


# Step 4: Main Execution and Analysis

In [15]:
def main():
    # Step 1: Load Dataset
    print("=" * 60)
    print("STEP 1: LOADING DATASET")
    print("=" * 60)
    
    dataset_path = r"C:\Users\hello\ML Projects\DS 5006\Lab 04\20news-bydate"
    loader = DatasetLoader(dataset_path)
    
    try:
        train_data, train_labels, test_data, test_labels = loader.load_dataset()
        
        if len(train_data) == 0:
            print("ERROR: No training data loaded!")
            return
        
        print(f"\n✓ Successfully loaded dataset:")
        print(f"  Training samples: {len(train_data)}")
        print(f"  Test samples: {len(test_data)}")
        print(f"  Categories: {len(loader.categories)}")
        
    except Exception as e:
        print(f"Error loading dataset: {e}")
        return
    
    print("\n" + "=" * 60)
    print("STEP 2: PREPROCESSING AND FEATURE EXTRACTION")
    print("=" * 60)
    
    print("Building vocabulary...")
    vocabulary = build_vocabulary(train_data, max_vocab_size=2000)
    print(f"Vocabulary size: {len(vocabulary)}")

    print("Converting text to vectors (using subsets for speed)...")
    sample_size = min(2000, len(train_data))  
    X_train = [text_to_vector(doc, vocabulary) for doc in train_data[:sample_size]]
    y_train = train_labels[:sample_size]
    X_test = [text_to_vector(doc, vocabulary) for doc in test_data[:min(500, len(test_data))]]
    y_test = test_labels[:min(500, len(test_data))]
    
    print("\n" + "=" * 60)
    print("STEP 3: TESTING DIFFERENT K VALUES")
    print("=" * 60)
    
    k_values = [1, 3, 5, 7, 9, 11]
    results = {}
    
    for k in k_values:
        print(f"Testing k={k}...")
        knn = KNNClassifier(k=k)
        knn.fit(X_train, y_train)
        predictions = knn.predict(X_test)
        accuracy = knn.accuracy(y_test, predictions)
        results[k] = accuracy
        print(f"  k={k}, Accuracy: {accuracy:.4f}")
    
    print("\n" + "=" * 60)
    print("STEP 4: TESTING WITH STOPWORDS REMOVAL")
    print("=" * 60)

    print("Building vocabulary WITHOUT stopwords...")
    vocabulary_no_sw = build_vocabulary(train_data, max_vocab_size=2000, remove_stopwords=True)
    
    print("Top 20 words in vocabulary (stopwords removed):")
    for i, word in enumerate(vocabulary_no_sw[:20]):
        print(f"{i+1}. {word}")
    
    X_train_no_sw = [text_to_vector(doc, vocabulary_no_sw, remove_stopwords=True) for doc in train_data[:sample_size]]
    X_test_no_sw = [text_to_vector(doc, vocabulary_no_sw, remove_stopwords=True) for doc in test_data[:min(500, len(test_data))]]
    
    results_no_sw = {}
    for k in k_values:
        print(f"Testing k={k} with stopwords removed...")
        knn = KNNClassifier(k=k)
        knn.fit(X_train_no_sw, y_train)
        predictions = knn.predict(X_test_no_sw)
        accuracy = knn.accuracy(y_test, predictions)
        results_no_sw[k] = accuracy
        print(f"  k={k}, Accuracy: {accuracy:.4f}")
    
    print("\n" + "=" * 60)
    print("STEP 5: RESULTS ANALYSIS")
    print("=" * 60)
    
    print("k\tWith Stopwords\tWithout Stopwords\tDifference")
    print("-" * 50)
    
    best_k = None
    best_accuracy = 0
    
    for k in k_values:
        diff = results_no_sw[k] - results[k]
        print(f"{k}\t{results[k]:.4f}\t\t{results_no_sw[k]:.4f}\t\t{diff:+.4f}")
        
        if results_no_sw[k] > best_accuracy:
            best_accuracy = results_no_sw[k]
            best_k = k
    
    print("=" * 50)
    print(f"Best performance: k={best_k}, Accuracy: {best_accuracy:.4f}")
    
    print("\n" + "=" * 60)
    print("STEP 6: FULL DATASET EVALUATION (OPTIONAL)")
    print("=" * 60)
    
    full_evaluation = input("Run evaluation on full dataset? (y/n): ").lower()
    if full_evaluation == 'y':
        print("Processing full dataset (this may take a while)...")
        X_train_full = [text_to_vector(doc, vocabulary_no_sw) for doc in train_data]
        y_train_full = train_labels
        X_test_full = [text_to_vector(doc, vocabulary_no_sw) for doc in test_data]
        y_test_full = test_labels
        
        knn = KNNClassifier(k=best_k)
        knn.fit(X_train_full, y_train_full)
        predictions = knn.predict(X_test_full) 
        accuracy = knn.accuracy(y_test_full, predictions)
        print(f"Full dataset accuracy (k={best_k}): {accuracy:.4f}")

if __name__ == "__main__":
    main()

STEP 1: LOADING DATASET
Discovering dataset structure...
Exploring directory structure...
Found directory: 20news-bydate-test
  Contains subdirectories: ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware']...
Found directory: 20news-bydate-train
  Contains subdirectories: ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware']...

Finding dataset paths...
Found training data at: C:\Users\hello\ML Projects\DS 5006\Lab 04\20news-bydate\20news-bydate-train
Found test data at: C:\Users\hello\ML Projects\DS 5006\Lab 04\20news-bydate\20news-bydate-test

Loading categories from: C:\Users\hello\ML Projects\DS 5006\Lab 04\20news-bydate\20news-bydate-train
Found 20 categories: ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.s

Run evaluation on full dataset? (y/n):  n


# OBSERVATIONS

1. By analysing k-nearest neighbour with and without stopwords, it turned out that KNN performs with **better accuracy when KNN classification is done without stopwords** (i.e. when stopwords are removed from vocabulary).
2. By varying k values, it came into observation that **as k value increases, accuracy for KNN without stopwords increases.**